# 민감도 분석 실습

**Sensitivity Analysis · OAT · Morris**

입력 변수가 출력에 얼마나 영향을 주는지 정량적으로 확인하는 분석.

소재 분야에서 이해하기: 온도와 시간 중 어느 조건이 수율을 더 좌우하는지 본다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [SALib 민감도 분석 문서](https://salib.readthedocs.io/en/latest/)

## 1. 한 번에 하나만 바꾸는 방법의 한계

OAT(one-at-a-time)는 상호작용을 놓칩니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def process(x):
    """x1 온도, x2 시간, x3 첨가비율 (모두 0-1로 정규화). 상호작용 항이 들어 있습니다."""
    x1, x2, x3 = x[:, 0], x[:, 1], x[:, 2]
    return 3 * x1 + 0.5 * x2 + 4 * x1 * x3 + 0.2 * x3

centre = np.array([[0.5, 0.5, 0.5]])
print('기준점 출력 %.3f' % process(centre)[0])

In [ ]:
sweep = np.linspace(0, 1, 50)
for index, name in enumerate(['temperature', 'time', 'additive']):
    points = np.repeat(centre, 50, axis=0)
    points[:, index] = sweep
    plt.plot(sweep, process(points), label=name)
plt.xlabel('normalised input'); plt.ylabel('output'); plt.legend(); plt.show()

for index, name in enumerate(['temperature', 'time', 'additive']):
    points = np.repeat(centre, 2, axis=0); points[:, index] = [0, 1]
    values = process(points)
    print('%-12s OAT 변화폭 %.3f' % (name, values[1] - values[0]))

## 2. 기준점을 옮기면 결론이 바뀝니다

In [ ]:
for base in ([0.1, 0.5, 0.1], [0.9, 0.5, 0.9]):
    base = np.array([base])
    print('기준점 %s' % base[0])
    for index, name in enumerate(['temperature', 'time', 'additive']):
        points = np.repeat(base, 2, axis=0); points[:, index] = [0, 1]
        values = process(points)
        print('   %-12s 변화폭 %.3f' % (name, values[1] - values[0]))
print('\n상호작용이 있으면 OAT 결과는 기준점에 의존합니다. 전역 민감도(소볼 지수)가 필요한 이유입니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#sensitivity-analysis)을 여세요.